In [ ]:
!pip install torch==2.2.0
!pip install torchtext==0.17.0
!pip install numpy==1.26.0

In [ ]:
!pip install kagglehub[pandas-datasets]

In [ ]:
import torch.nn as nn
import torch
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator,GloVe
import kagglehub
from kagglehub import KaggleDatasetAdapter
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split
import time
from sklearn.model_selection import train_test_split
import math

In [ ]:
dataset = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    'thedevastator/uncovering-enron-employees-secrets-exploring-the',
    'test.csv'
)

In [ ]:
email_body = dataset['email_body']
subject = dataset['subject_line']

In [ ]:
tokenizer = get_tokenizer('basic_english')

In [ ]:
def yield_tokens(chars):
  for c in chars:
    yield tokenizer(c)

In [ ]:
vocab = build_vocab_from_iterator(yield_tokens(email_body),specials=['<unk>'])
vocab.set_default_index(vocab['<unk>'])

In [ ]:
def process_text(data):
  final_data = []

  for d in data:
    final_data.append(torch.tensor(vocab(tokenizer(d)),dtype=torch.long))

  final_data = pad_sequence(final_data,batch_first=True)

  return final_data

In [ ]:
email_body = process_text(list(email_body))

In [ ]:
subject = process_text(list(subject))

In [ ]:
subject.shape

In [ ]:
len(subject)

In [ ]:
# Use a smaller padding value for the subject tensor to match the transformer's input size
subject = pad_sequence([nn.functional.pad(s,(0,1885-21),'constant',0) for s in subject],batch_first=True,padding_value=0)

In [ ]:
dataset = torch.utils.data.TensorDataset(email_body,subject)

In [ ]:
for data,label in dataset:
  print(data)
  print(label)
  break

In [ ]:
train_dataset,test_dataset = train_test_split(dataset,test_size=0.2)

In [ ]:
train_dataloader = DataLoader(train_dataset,batch_size=64,shuffle=True)
test_dataloader = DataLoader(test_dataset,batch_size=64,shuffle=False)

In [ ]:
for text,label in train_dataloader:
  print(text)
  print(label)
  break

In [ ]:
transformer = nn.Transformer(d_model=1885,nhead=5,num_encoder_layers=6,num_decoder_layers=6,activation='relu',batch_first=True,dim_feedforward=1885).to(device)

In [ ]:
transformer.decoder.layers[0]

In [ ]:
class FeatureAdapter(nn.Module):
  def __init__(self,model_dim,bottle_neck):
    super(FeatureAdapter,self).__init__()
    self.model_dim = model_dim
    self.bottle_neck = bottle_neck
    self.bottle_neck_transform = nn.Sequential(
        nn.Linear(self.model_dim,self.bottle_neck),
        nn.ReLU(),
        nn.Linear(self.bottle_neck,self.model_dim)
    )
  def forward(self,x):
    out_transform = self.bottle_neck_transform(x)
    output_with_residual = x  + out_transform
    return output_with_residual

In [ ]:
class Adapted(nn.Module):
  def __init__(self,linear):
    super().__init__()
    self.linear = linear
    model_dim = self.linear.out_features
    bottle_neck_features = model_dim//2
    self.adaptor = FeatureAdapter(model_dim,bottle_neck_features)

  def forward(self,x):
    x = self.linear(x)
    x = self.adaptor(x)
    return x

In [ ]:
class LoRA_Layer(nn.Module):
  def __init__(self,rank,alpha,in_features,out_features):
    super().__init__()
    standard_deviation = 1/torch.sqrt(torch.tensor(rank).float())
    self.A = nn.Parameter(torch.randn(in_features,rank) * standard_deviation)
    self.B = nn.Parameter(torch.zeros(rank,in_features))
    self.alpha = alpha
    self.rank = rank

  def forward(self,x):
    return (self.alpha/self.rank) * (x @ self.A @ self.B)

In [ ]:
class LoRA_DenseLayer(nn.Module):
  def __init__(self,linear,rank,alpha):
    super().__init__()
    self.linear =linear
    self.lora = LoRA_Layer(rank,alpha,self.linear.in_features,self.linear.out_features)

  def forward(self,x):
    return self.linear(x) + self.lora(x)

In [ ]:

transformer.decoder.layers[0].linear2

In [ ]:
transformer.parameters

In [ ]:
transformer.encoder.layers[0].linear2

In [ ]:
for n in range(6):
  transformer.encoder.layers[n].linear1 = LoRA_DenseLayer(transformer.encoder.layers[n].linear1,2,0.1).to(device)
  transformer.encoder.layers[n].linear2 = LoRA_DenseLayer(transformer.encoder.layers[n].linear2,2,0.1).to(device)
  transformer.decoder.layers[n].linear1 = LoRA_DenseLayer(transformer.decoder.layers[n].linear1,2,0.1).to(device)
  transformer.decoder.layers[n].linear2 = LoRA_DenseLayer(transformer.decoder.layers[n].linear2,2,0.1).to(device)

In [ ]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(transformer.parameters(),lr=1)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer,1.0,gamma=0.1)

In [ ]:
epochs = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
for text,label in train_dataloader:
  print(text.shape)
  print(label.shape)
  break

In [ ]:
with torch.no_grad():
  for text,label in train_dataloader:
    text,label = text.to(device),label.to(device)
    print(type(text))
    #output = transformer(text,label)
    break

In [ ]:
# Add an embedding layer to convert token indices to embeddings
embedding = nn.Embedding(len(vocab), 1885).to(device)

for epoch in range(epochs):
  transformer.train()
  cum_loss = 0
  start  = time.time()
  for text,label in train_dataloader:
    optimizer.zero_grad()
    label,text  = label.to(device),text.to(device)

    # Embed the input tensors
    text_embedded = embedding(text)
    label_embedded = embedding(label)

    predicted_label = transformer(text_embedded,label_embedded)
    loss = criterion(predicted_label,label_embedded) # Calculate loss with embedded labels
    loss.backward()
    torch.nn.utils.clip_grad_norm_(transformer.parameters(),0.1)
    optimizer.step()
    scheduler.step()
    cum_loss += loss.item()
    print("Step completed")
  print("Time taken: ",time.time()-start)
  print(" Cumulative Loss: ",cum_loss)
  print("Acc: ",evaluate())

In [ ]:
def evaluate():
  transformer.eval()
  total_acc,total_count = 0,0

  for text,label in test_dataloader:
    text,label = text.to(device),label.to(device)
    output = transformer(text)
    predicted = torch.max(output.data,1)[1]
    total_acc += (predicted == label).sum().item()
    total_count += len(label)

  return total_acc/total_count

In [ ]:
def predict():
  user_inp = input("Enter the body of the email: ")
  user_inp = torch.tensor(vocab(tokenizer(user_inp),dtype=torch.long)).unsqueeze(0)
  user_inp = user_inp.to(device)
  output = model(user_inp)
  predicted = torch.max(output.data,1)[1]